# Getting Started with catsim

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/01_getting_started.ipynb)

**For:** researchers who know the basics of item response theory and are new to catsim.
**Level:** beginner.

You will build an item bank, generated or from your own calibrated parameters, see what its items
measure, give a computerized adaptive test to 500 simulated examinees, read the results, follow one
examinee's test item by item, and repeat the run from its seed. The
[introduction](https://douglasrizzo.com.br/catsim/introduction.html) explains adaptive testing, and
[Terms and notation](https://douglasrizzo.com.br/catsim/terms-and-notation.html) the terms used here.

To run the notebook outside this repository, install catsim with its plot extra, which adds
matplotlib: `pip install 'catsim[plot]'`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from catsim import CatEngine, ItemBank, SimulationRunner, plot
from catsim.estimation import EAPEstimator
from catsim.initialization import FixedPointInitializer
from catsim.selection import MaxInfoSelector
from catsim.stopping import MaxLengthStopper, StandardErrorStopper, StopRule

## One seed for the notebook

A simulation draws random numbers for three things: the item bank, the examinees' true abilities
and the run itself, which simulates each response. `SEED` is the notebook's only seed.
`SeedSequence(SEED).spawn(3)` splits it into three independent seeds, one for each, so every cell
below gives the same numbers each time it runs, and another `SEED` gives another bank, population
and run.

In [ ]:
SEED = 2026
bank_seed, population_seed, run_seed = np.random.SeedSequence(SEED).spawn(3)

## An item bank

An item bank holds the calibrated parameters of its items, one row per item: discrimination $a$,
difficulty $b$, guessing parameter $c$ and upper asymptote $d$. `ItemBank.generate` draws a bank of
300 items of the three-parameter logistic model (3PL), whose upper asymptotes are all 1, for the
simulation below.

In [ ]:
item_bank = ItemBank.generate(300, model="3PL", rng=np.random.default_rng(bank_seed))
print(item_bank)
print(item_bank.items[:3].round(2))

To use a bank you calibrated, pass its parameters to `ItemBank` as an array with one row per item.
From a CSV file, that array is `np.loadtxt("items.csv", delimiter=",")`; here five 3PL items are
written out, with the columns $a$, $b$ and $c$. A three-column array is read as $a$, $b$, $c$ with
$d = 1$ (two columns as $a$, $b$; one as $b$).

catsim's logistic models have no scaling constant. If your calibration reports discriminations for a
model with $D = 1.7$, multiply them by 1.7. If it reports an intercept rather than a difficulty, as
mirt does by default, the difficulty is $b = -\text{intercept} / a$.

In [ ]:
calibrated = np.array([
  [0.8, -1.5, 0.20],
  [1.2, -0.5, 0.15],
  [1.9, 0.0, 0.25],
  [1.0, 0.7, 0.10],
  [1.5, 1.6, 0.20],
])
my_bank = ItemBank(calibrated)
print(my_bank)
print(my_bank.items)

## What an item measures

An item's characteristic curve gives the probability of a correct response at each ability, and
its information curve how much a response to it sharpens an ability estimate there.
`plot.item_curve` draws both for two of the calibrated items: item 0, easy and weakly
discriminating, and item 2, of middle difficulty and the most discriminating.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot.item_curve(my_bank[0], kind="both", ax=axes[0], title="Item 0")
plot.item_curve(my_bank[2], kind="both", ax=axes[1], title="Item 2")
plt.tight_layout()
plt.show()

Each characteristic curve (blue, left axis) rises from the guessing parameter, 0.2 and 0.25, toward
1, most steeply near the difficulty. Item 2's larger discrimination makes its curve steeper and its
information (red, right axis) peak about five times higher, at about 0.56 against 0.11, but over a
narrower range of abilities. The dot marks each peak, a little above the difficulty: with guessing,
a correct response says less about an examinee of low ability. An adaptive test gives each item to
the examinees near its peak.

## A simulation

A test is an item bank and four strategies, which a `CatEngine` holds:

- an initializer gives the ability estimate before the first response, here 0;
- a selector chooses each item, here the most informative one at the current estimate;
- an estimator updates the estimate after each response, here the expected a posteriori (EAP)
  estimate, which starts from a standard normal prior;
- a stopper ends the test, here once the standard error of the estimate is at most `TARGET_SE`, or
  after `MAX_ITEMS` items, whichever comes first.

The length cap ends the test of an examinee whose target precision the bank cannot reach. A
`SimulationRunner` gives the test to each of `N_EXAMINEES` examinees, whose true abilities are drawn
from a standard normal distribution, and returns a `SimulationResult`.

In [ ]:
TARGET_SE = 0.4
MAX_ITEMS = 30
N_EXAMINEES = 500

stopper = StopRule(StandardErrorStopper(TARGET_SE), MaxLengthStopper(MAX_ITEMS))
engine = CatEngine(item_bank, FixedPointInitializer(0.0), MaxInfoSelector(), EAPEstimator(), stopper)
true_thetas = np.random.default_rng(population_seed).normal(0, 1, N_EXAMINEES)
result = SimulationRunner(engine).run(true_thetas, rng=np.random.default_rng(run_seed))

In [ ]:
lengths = result.test_lengths
short_of_target = result.final_ses > TARGET_SE
print(f"bias {result.bias:+.3f}, RMSE {result.rmse:.3f}")
print(f"test length: shortest {lengths.min()}, mean {lengths.mean():.1f}, longest {lengths.max()}")
print(f"examinees who ended above the target SE: {np.count_nonzero(short_of_target)}")
print(f"their true abilities: {np.sort(result.true_thetas[short_of_target]).round(2)}")
print(f"highest exposure rate {result.exposure_rates.max():.2f}")
print(f"items never given: {np.count_nonzero(result.exposure_rates == 0)} of {item_bank.n_items}")
print(f"overlap rate {result.overlap_rate}")

The bias, the mean of the estimates' errors, is close to 0, and the RMSE, 0.42, is close to the 0.4
target: the standard errors the estimator reports match the errors it makes. Tests take 14 to 30
items, 18 on average. Ten examinees ended with a standard error still above the target, so their
tests ran to the 30-item cap; nine of them have true abilities 1.8 or more from the middle of the population, where the
bank has few informative items.

The highest exposure rate is 1: every test starts at the same estimate, so every examinee gets the
same first item. 195 of the 300 items are never given, because maximum information keeps choosing
the most discriminating items. The overlap rate is None because the tests have different lengths.
The [Running simulations](https://douglasrizzo.com.br/catsim/running-simulations.html) guide lists everything a result holds.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(result.true_thetas, result.final_thetas, s=10, alpha=0.6)
ax.axline((0, 0), slope=1, color="gray", linestyle="--")
ax.set_xlabel("true ability θ")
ax.set_ylabel("final estimate θ̂")
plt.show()

Each point is one examinee. The points scatter around the diagonal, where the estimate equals the
true ability. The errors grow toward the ends of the scale, and there most estimates lie closer to 0
than the true ability: the EAP estimate is pulled toward the mean of its prior, the more so the less
the responses say. The next notebook compares estimators.

## One examinee's test

`result.sessions` holds each examinee's test, in the order of `true_thetas`: the items given, the
responses (1 for correct) and the estimate before the first item and after each response.
`plot.session_progress` draws it.

In [ ]:
session = result.sessions[0]
print(f"true ability {session.true_theta:.2f}, final estimate {session.current_theta:.2f}, SE {session.current_se:.2f}")
print("items:", session.administered_items)
print("responses:", [int(response) for response in session.responses])
print("estimates:", np.round(session.theta_history, 2))
plot.session_progress(session, item_bank)
plt.show()

The estimate (blue) starts at 0 and moves most in the first items, when one response changes it a
lot: up after two correct responses, down after two incorrect ones. It then settles, and ends within
one standard error of the true ability (dashed), 0.54. The standard error (green, right axis) falls
with almost every response, from about 0.9 after the first item to 0.39 after the 16th, when the test stops
for reaching the target, and the shaded band of one standard error on each side narrows with it. The
item difficulties (orange) stay near the estimate, mostly a little below it: maximum information
chooses the item whose information peaks at the estimate, and an item's peak lies a little above its
difficulty.

## Repeating the run

A generator built from the same seed repeats every test. A run without `rng` draws fresh random
numbers, so its tests differ every time and cannot be repeated. The
[Reproducibility](https://douglasrizzo.com.br/catsim/reproducibility.html) guide shows how to seed a study of many runs.

In [ ]:
runner = SimulationRunner(engine)
again = runner.run(true_thetas, rng=np.random.default_rng(run_seed))
unseeded = runner.run(true_thetas)
print("same seed, same estimates:", np.array_equal(again.final_thetas, result.final_thetas))
print("no seed, same estimates:", np.array_equal(unseeded.final_thetas, result.final_thetas))

## Where to go next

- [02_comparing_strategies.ipynb](02_comparing_strategies.ipynb) compares estimators, selectors,
  exposure control and stoppers over replications.
- The [Running simulations](https://douglasrizzo.com.br/catsim/running-simulations.html) guide covers this notebook's tasks as
  a reference.
- The [technique pages](https://douglasrizzo.com.br/catsim/api/techniques/index.html) define each method and say when to
  choose it.